# 1 Importing Packages

How the *MESSAGE_CA* class works: the class will contain all the functions necessary to run the MESSAGEix Canada model. All that you need to run a model is to:

1. Create your scenario folder under `model/scenarios` with its own `config.toml` file

2. Declare the class reading that scenario, for example: `platform, msg, msg_folder = MESSAGE_CA(scenario_name='baseline')`

In [1]:
import os
import sys


In [3]:
from model.Model import model_MSG 
#from model.Plotter import plotter_MSG


# Autoreload modules when changiges are applied to them
%load_ext autoreload 
%autoreload all

In [ ]:
model_MSG.clone_scenario

In [2]:
# Sys and OS Operations

os.chdir("../../")

# Scenario Creation and Set-up

Setting up the MESSAGEix scenario using the scenario configuration file and an input file for technology options in technology.yaml

In [44]:
# Set-upmsg = model_MSG(scenario_name=scenario, config=None)
msg = model_MSG(scenario_name='baseline', config=None)
platform, msg_scenario, msg_folder = msg.setup_scenario()


Model Logs: Setting up scenario


		Model creation time: 1 Min and 19 Sec 		
Model Logs: Solving the model...


				Solving process time: 3 Min and 29 Sec				


In [45]:
msg_scenario.remove_solution()
msg_scenario.check_out()

In [ ]:
'''import pandas as pd

def sync_var_costs(msg_scenario):
    """Sync var_cost entries from fix_cost combinations."""
    
    # Load parameters
    fix_cost = msg_scenario.par("fix_cost")
    var_cost = msg_scenario.par("var_cost")
    
    # Get unique combinations
    fix_combo = fix_cost[["node_loc", "technology", "year_act", "year_vtg"]].drop_duplicates()
    var_combo = var_cost[["node_loc", "technology", "year_act", "year_vtg"]].drop_duplicates()
    
    # Find missing combinations
    new_entries = fix_combo.merge(
        var_combo, 
        how="left", 
        indicator=True
    ).query("_merge == 'left_only'").drop("_merge", axis=1)
    
    if not new_entries.empty:
        # Create new var_cost entries
        new_var_cost = new_entries.assign(
            mode="M1",
            time="year",
            unit=fix_cost["unit"].iloc[0],  # Inherit units from fix_cost
            value=0.0,
        )
        
        msg_scenario.add_par("var_cost", new_var_cost)
        msg_scenario.commit("Added missing var_cost entries from fix_cost")
    
    return len(new_entries)'''





In [6]:
# Alberta, SMR (no CCS)
msg_scenario.add_par('historical_activity', 
    {
        'node_loc': ['Alberta'],
        'year_act': [2020],
        'technology': ['h2_smr'],
        'mode': ['M1'],
        'time': ['year'],
        'value': [6.5],  # GWa
        'unit': ['GWa'],
    })

# Ontario, SMR
msg_scenario.add_par('historical_activity', 
    {
        'node_loc': ['Ontario'],
        'year_act': [2020],
        'technology': ['h2_smr'],
        'mode': ['M1'],
        'time': ['year'],
        'value': [2.0],
        'unit': ['GWa'],
    })

# Quebec, SMR
msg_scenario.add_par('historical_activity', 
    {
        'node_loc': ['Quebec'],
        'year_act': [2020],
        'technology': ['h2_smr'],
        'mode': ['M1'],
        'time': ['year'],
        'value': [0.08],
        'unit': ['GWa'],
    })

# Quebec, Electrolysis
msg_scenario.add_par('historical_activity', 
    {
        'node_loc': ['Quebec'],
        'year_act': [2020],
        'technology': ['h2_elec'],
        'mode': ['M1'],
        'time': ['year'],
        'value': [0.02],
        'unit': ['GWa'],
    })

# British Columbia, SMR
msg_scenario.add_par('historical_activity', 
    {
        'node_loc': ['BritishColumbia'],
        'year_act': [2020],
        'technology': ['h2_smr'],
        'mode': ['M1'],
        'time': ['year'],
        'value': [0.3],
        'unit': ['GWa'],
    })

# Manitoba, Electrolysis
msg_scenario.add_par('historical_activity', 
    {
        'node_loc': ['Manitoba'],
        'year_act': [2020],
        'technology': ['h2_elec'],
        'mode': ['M1'],
        'time': ['year'],
        'value': [0.01],
        'unit': ['GWa'],
    })

# Nova Scotia, Electrolysis
msg_scenario.add_par('historical_activity', 
    {
        'node_loc': ['NovaScotia'],
        'year_act': [2020],
        'technology': ['h2_elec'],
        'mode': ['M1'],
        'time': ['year'],
        'value': [0.01],
        'unit': ['GWa'],
    })

In [ ]:
added_count = sync_var_costs(msg_scenario)

In [15]:
df=msg_scenario.var("ACT")

In [27]:
df=msg_scenario.par("historical_activity", filters={"node_loc": "Alberta", "technology":"gas_exp", "year_act":["2015","2020"]})

In [28]:
df.loc[df["year_act"]==2015,"value"]-=38.1738155429925

In [30]:
df.loc[df["year_act"]==2020,"value"]-=45.26967673067

In [16]:
df

,node_loc,technology,year_vtg,year_act,mode,time,lvl,mrg
0,Alberta,BCA_AgWasteEM,2010,2010,P,year,0.0,0.000000e+00
1,Alberta,BCA_AgWasteEM,2015,2015,P,year,0.0,0.000000e+00
2,Alberta,BCA_AgWasteEM,2020,2020,P,year,0.0,0.000000e+00
3,Alberta,BCA_AgWasteEM,2025,2025,P,year,0.0,4.940656e-324
4,Alberta,BCA_AgWasteEM,2030,2030,P,year,0.0,4.940656e-324
...,...,...,...,...,...,...,...,...
105196,Yukon,h2_gas_fc_trp,2050,2055,M1,year,0.0,1.595031e+00
105197,Yukon,h2_gas_fc_trp,2050,2060,M1,year,0.0,2.923443e+00
105198,Yukon,h2_gas_fc_trp,2055,2055,M1,year,0.0,1.595031e+00
105199,Yukon,h2_gas_fc_trp,2055,2060,M1,year,0.0,2.923443e+00


In [15]:
msg_scenario.add_par("bound_activity_lo", df)

In [15]:
import pandas as pd
import numpy as pd

In [21]:
def _calculate_demand(scenario, reg, years, changing_first_model_year):
    # Retrieve current demands
    df = (
        scenario.par("demand", {"node": reg})
        .drop_duplicates(["commodity", "level"])
        .set_index(["commodity", "level"])
    )

    dem = pd.DataFrame(
        columns=list(scenario.par("demand").columns), index=product(df.index, years)
    )

    dem = dem.assign(node=reg, time="year", unit="GWa").reset_index()
    dem["commodity"] = dem.apply(lambda r: r["index"][0][0], axis=1)
    dem["level"] = dem.apply(lambda r: r["index"][0][1], axis=1)
    dem["year"] = dem.apply(lambda r: int(r["index"][1]), axis=1)
    dem = dem.drop("index", axis=1)

    for i in dem.index:
        tmp = dem.loc[[i]]
        # Reading data for demand from the model
        # List of technologies feeding into this demand commodity
        tec_list = list(
            set(
                (
                    scenario.par(
                        "output",
                        {
                            "level": tmp.level,
                            "node_loc": reg,
                            "commodity": tmp.commodity,
                        },
                    )
                )["technology"]
            )
        )

        # Summing historical activity of these technologies

        if changing_first_model_year is False:
            par = "historical_activity"
        else:
            par = "bound_activity_lo"
        hist_act = sum(
            scenario.par(
                par,
                {"technology": tec_list, "year_act": tmp.year, "node_loc": reg},
            )["value"]
        )
        tmp["value"] = hist_act
        dem = tmp.combine_first(dem)
    dem = dem.reset_index(drop=True)

    #    for i in dem.index:
    #        dem.loc[[i], "commodity"] = i[0][0]
    #        dem.loc[[i], "level"] = i[0][1]
    #        dem.loc[[i], "year"] = int(i[1])
    #        dem["node"] = reg
    #        dem["time"] = "year"
    #        dem["unit"] = "GWa"
    #
    #        # Reading data for demand from the model
    #        # List of technologies feeding into this demand commodity
    #        tec_list = list(
    #            set(
    #                (
    #                    scenario.par(
    #                        "output",
    #                        {"level": i[0][1], "node_loc": reg, "commodity": i[0][0]},
    #                    )
    #                )["technology"]
    #            )
    #        )
    #
    #        # Summing historical activity of these technologies
    #
    #        if changing_first_model_year is False:
    #            hist_act = sum(
    #                scenario.par(
    #                    "historical_activity",
    #                    {"technology": tec_list, "year_act": i[1], "node_loc": reg},
    #                )["value"]
    #            )
    #            dem.loc[[i], "value"] = hist_act
    #
    #        else:
    #            hist_act = sum(
    #                scenario.par(
    #                    "bound_activity_lo",
    #                    {"technology": tec_list, "year_act": i[1], "node_loc": reg},
    #                )["value"]
    #            )
    #            dem.loc[[i], "value"] = hist_act
    #
    #    dem = dem.reset_index(drop=True)

    if not dem.empty:
        print(
            '> Parameter "demand" in "' + reg + '" was calibrated to'
            " the historical data from the model."
        )
    else:
        print('> There is no historical data in "' + reg + '" !!!')

    return dem

In [ ]:
# Ensure these packages are installed
import pandas as pd
import message_ix
from itertools import product

changing_first_model_year = False  # Set to True if modifying first model year


In [23]:
years = [2020, 2025, 2030, 2035, 2040, 2045, 2050]
import pandas as pd

regions = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

all_dfs = []  # Initialize an empty list to store dataframes

for reg in regions:
    # Retrieve old demands
    df_old = msg_scenario.par("demand", {"node": reg, "year": years})

    # Create df with recalculated demands
    df = _calculate_demand(msg_scenario, reg, years, changing_first_model_year)

    df["unit"] = "GWa"
    df = df.round(2)

    # Append to the list
    all_dfs.append(df)

# Concatenate all DataFrames into one
final_df = pd.concat(all_dfs, ignore_index=True)
rate_of_change = {
    2025: 0.052566283,
    2030: 0.049920952,
    2035: 0.045423593,
    2040: 0.044337335,
    2045: 0.040088765,
    2050: 0.038095737,
    2055: 0.038278554,
    2060: 0.03846426
}

# Ensure `final_df` has the correct structure and only contains 2020 values
df_2020 = final_df[final_df['year'] == 2020].copy()

# Set index for better manipulation
df_2020.set_index(["node", "commodity", "level"], inplace=True)

# Create projections based on rate of change
projections = []
for (node, commodity, level), row in df_2020.iterrows():
    base_value = row["value"]
    for year, rate in rate_of_change.items():
        projected_value = base_value * (1 + rate)  # Applying growth rate
        projections.append([node, commodity, level, year, "year", projected_value, row["unit"]])

# Convert projections to DataFrame with correct indexing
df_projections = pd.DataFrame(projections, columns=["node", "commodity", "level", "year", "time", "value", "unit"])

# Reset index for merging
df_2020.reset_index(inplace=True)

# Combine original and projected data
df_final = pd.concat([df_2020, df_projections]).reset_index(drop=True)

> Parameter "demand" in "Ontario" was calibrated to the historical data from the model.
> Parameter "demand" in "Quebec" was calibrated to the historical data from the model.
> Parameter "demand" in "BritishColumbia" was calibrated to the historical data from the model.
> Parameter "demand" in "Alberta" was calibrated to the historical data from the model.
> Parameter "demand" in "Manitoba" was calibrated to the historical data from the model.
> Parameter "demand" in "Saskatchewan" was calibrated to the historical data from the model.
> Parameter "demand" in "NovaScotia" was calibrated to the historical data from the model.
> Parameter "demand" in "NewBrunswick" was calibrated to the historical data from the model.
> Parameter "demand" in "NewfoundlandandLabrador" was calibrated to the historical data from the model.
> Parameter "demand" in "PrinceEdwardIsland" was calibrated to the historical data from the model.
> Parameter "demand" in "NorthwestTerritories" was calibrated to the his

In [29]:

msg_scenario.add_par("demand", df_final)  # "demand" must be a string literal


In [ ]:
fix_cost_data = msg_scenario.par('fix_cost')
filtered_data = fix_cost_data[(fix_cost_data['technology'] == 'gas_ppl') & 
                              (fix_cost_data['year_act'].isin([2020, 2025, 2040])) & (fix_cost_data['year_vtg'].isin([2020, 2025, 2040]))]


print(filtered_data)


In [ ]:
import pandas as pd
import pyam as pyam
import message_ix as message_ix

In [ ]:
node_commodity=[("Alberta", "primary", "gas", 2020)]


In [ ]:

for node, lvl, com, yr in node_commodity:
    # Demand
    dem = msg_scenario.par(
        "demand", {"commodity": com, "node": node, "level": lvl, "year": yr}
    )
    dem = float(dem["value"])
    # Activity
    out = msg_scenario.par("output", {"commodity": com, "node_loc": node, "year_act": yr})
    tec_list = list(set(out["technology"]))
    act = msg_scenario.par(
        "historical_activity", {"technology": tec_list, "node_loc": node, "year_act": yr}
    )["value"].sum()

In [16]:
years = [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060]

provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

In [11]:
msg_scenario.add_set("storage_tec", "h2_stor_geo")

msg_scenario.add_set("level", "storage")

msg_scenario.add_set("level_storage", "storage")

In [17]:
#Adding hydrogen storage with seasonality

#msg_scenario.add_set("storage_tec", "h2_stor_geo")

#msg_scenario.add_set("level_storage", "storage")


time_steps = ["winter", "spring", "summer", "fall"]
msg_scenario.add_set("time", time_steps)

time_level = "season"
msg_scenario.add_set("lvl_temporal", time_level)

for t in time_steps:
    msg_scenario.add_set("map_temporal_hierarchy", [time_level, t, "year"])

for t in time_steps:
    msg_scenario.add_par("duration_time", [t], 0.25, "-")

def yearly_to_season(scen, parameter, data, tech_list, filters=None):
    if filters:
        old = msg_scenario.par(parameter, filters)
    else:
        old = msg_scenario.par(parameter)
    
    # Filter for specified technologies
    old = old[old.technology.isin(tech_list)]
    
    msg_scenario.remove_par(parameter, old)

    time_idx = [x for x in scen.idx_names(parameter) if "time" in x]
    for tech in tech_list:
        if tech in data[parameter]:
            tech_data = old[old.technology == tech].copy()
            for h in data[parameter][tech].keys():
                new = tech_data.copy()
                for time in time_idx:
                    new[time] = h
                new["value"] = data[parameter][tech][h] * new["value"]
                msg_scenario.add_par(parameter, new)

tech_list = ["h2_stor_geo", "h2_stor_geo_charge", "h2_stor_geo_discharge"]

seasonal_data = {
    "input": {tech: {'winter': 1, 'spring': 1, 'summer': 1, 'fall': 1} for tech in tech_list},
    "output": {tech: {'winter': 1, 'spring': 1, 'summer': 1, 'fall': 1} for tech in tech_list},
    "capacity_factor": {tech: {'winter': 1, 'spring': 1, 'summer': 1, 'fall': 1} for tech in tech_list},
    "var_cost": {tech: {'winter': 1, 'spring': 1, 'summer': 1, 'fall': 1} for tech in tech_list}
}

# For input parameter
yearly_to_season(msg_scenario, "input", seasonal_data, tech_list)

# For output parameter
yearly_to_season(msg_scenario, "output", seasonal_data, tech_list)

# For cf parameter
yearly_to_season(msg_scenario, "capacity_factor", seasonal_data, tech_list)

# For var cost parameter
yearly_to_season(msg_scenario, "var_cost", seasonal_data, tech_list)\

for province in provinces:
    msg_scenario.add_set("map_tec_storage", [province, "h2_stor_geo_charge", "M1", "h2_stor_geo", "M1", "storage", "hydrogen", "season"])
    msg_scenario.add_set("map_tec_storage", [province, "h2_stor_geo_discharge", "M1", "h2_stor_geo", "M1", "storage", "hydrogen", "season"])

for year in years:
    for province in provinces:
        for step in time_steps:
            msg_scenario.add_par("storage_self_discharge", 
                 {"node": [province], 
                  "technology": ["h2_stor_geo"], 
                  "commodity": ["hydrogen"], 
                  "level": ["storage"],
                  "mode": ["M1"],
                  "year": [year],
                  "time": step, #slice
                  "value": [0.001]},  # Include the value in the dictionary
                 unit="-")

for province in provinces: 
    for slep in time_steps:  
        msg_scenario.add_par('storage_initial', 
                    {
                        'node': province,
                        'technology': "h2_stor_geo",
                        'year': 2020,
                        'mode': "M1",
                        'commodity': ["hydrogen"],
                        'level': ["storage"],
                        'time': "winter",
                        'value': 0.05, #constrain this
                        'unit': '-',
                    })

In [ ]:
msg_scenario.add_set("mode", "charge")
msg_scenario.add_set("mode", "discharge")
msg_scenario.add_set('level', 'storage')


In [33]:
msg_scenario.check_out()

In [ ]:
years = [2020, 2025, 2030, 2035, 2040, 2045, 2050]
import pandas as pd

provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

for province in provinces: 
    for year in years:  
        '''msg_scenario.add_par('relation_activity', 
                    {
                        'relation': ["CO2_cc"],
                        'node_rel': province,
                        'year_rel': year,
                        'node_loc': province,
                        'year_act': year,
                        'technology': ['coal_ppl_u'],
                        'mode': ["M1"],
                        'value': [2.14210526315789], #constrain this
                        'unit': ['-'],
                    })'''
         
        msg_scenario.add_par('relation_activity', 
                    {
                        'relation': ["CO2_cc"],
                        'node_rel': province,
                        'year_rel': [2030],
                        'node_loc': province,
                        'year_act': [2030],
                        'technology': ['oil_extr_7'],
                        'mode': ["M1"],
                        'value': [0.42], #constrain this
                        'unit': ['-'],
                    })
        msg_scenario.add_par('relation_activity', 
                    {
                        'relation': ["CO2_cc"],
                        'node_rel': province,
                        'year_rel': year,
                        'node_loc': province,
                        'year_act': year,
                        'technology': ['gas_cc'],
                        'mode': ["M1"],
                        'value': [0.860714285714286], #constrain this
                        'unit': ['-'],
                    })
        
        msg_scenario.add_par('relation_activity', 
                    {
                        'relation': ["CO2_cc"],
                        'node_rel': province,
                        'year_rel': year,
                        'node_loc': province,
                        'year_act': year,
                        'technology': ['gas_cc_ccs'],
                        'mode': ["M1"],
                        'value': [0.1], #constrain this
                        'unit': ['-'],
                    })
        
        msg_scenario.add_par('relation_activity', 
                    {
                        'relation': ["CO2_cc"],
                        'node_rel': province,
                        'year_rel': year,
                        'node_loc': province,
                        'year_act': year,
                        'technology': ['gas_ct'],
                        'mode': ["M1"],
                        'value': [0.705], #constrain this
                        'unit': ['-'],
                    })
        
        msg_scenario.add_par('relation_activity', 
                    {
                        'relation': ["CO2_cc"],
                        'node_rel': province,
                        'year_rel': year,
                        'node_loc': province,
                        'year_act': year,
                        'technology': ['gas_ppl'],
                        'mode': ["M1"],
                        'value': [1.205], #constrain this
                        'unit': ['-'],
                    })


In [ ]:
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

years = list(range(2020, 2061, 5))

In [ ]:
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

years = list(range(2020, 2061, 5))

In [ ]:
from message_ix import make_df

In [7]:
#Trade Hub
#change to lh2 commodity

provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

years = list(range(2020, 2061, 5))



# ========================
# 2. Add Nodes and Levels
# ========================
msg_scenario.add_set("node", ["Trade_hub", "global_trade"])
msg_scenario.add_set("level", ["trade"]) 

# ========================
# 3. Define Technologies
# ========================
# Provincial technologies
province_tecs = ["lh2_imp", "lh2_exp"]
# Hub-to-global technology
hub_tec = "lh2_global_exp"

msg_scenario.add_set("technology", province_tecs + [hub_tec])

# ======================
# 1. Provincial Importers
# ======================
for year_act in years:
    for year_vtg in years:
        if year_vtg <= year_act <= year_vtg + 40:
            for province in provinces:
                # lh2_imp: Trade_hub -> Province
                msg_scenario.add_par("input", {
                    "node_loc": [province],
                    "node_origin": "Trade_hub",
                    "technology": "lh2_imp",
                    "commodity": "lh2",
                    "level": "trade",  # Input from trade level
                    "year_vtg": [year_vtg],
                    "year_act": [year_act],
                    "mode": "M1",
                    "time": "year",
                    "time_origin": "year",
                    "value": 1,
                    "unit": "-"
                })
                
                msg_scenario.add_par("output", {
                    "node_loc": [province],
                    "node_dest": [province],
                    "technology": "lh2_imp",
                    "commodity": "lh2",
                    "level": "secondary",  # Output to provincial level
                    "year_vtg": [year_vtg],
                    "year_act": [year_act],
                    "mode": "M1",
                    "time": "year",
                    "time_dest": "year",
                    "value": 1,
                    "unit": "-"
                })

# ======================
# 2. Provincial Exporters 
# ======================
for year_act in years:
    for year_vtg in years:
        if year_vtg <= year_act <= year_vtg + 40:
            for province in provinces:
                # lh2_exp: Province -> Trade_hub
                msg_scenario.add_par("input", {
                    "node_loc": [province],
                    "node_origin": [province],
                    "technology": "lh2_exp",
                    "commodity": "lh2",
                    "level": "secondary",  # Takes from provincial level
                    "year_vtg": [year_vtg],
                    "year_act": [year_act],
                    "mode": "M1",
                    "time": "year",
                    "time_origin": "year",
                    "value": 1,
                    "unit": "-"
                })
                
                msg_scenario.add_par("output", {
                    "node_loc": [province],
                    "node_dest": "Trade_hub",
                    "technology": "lh2_exp",
                    "commodity": "lh2",
                    "level": "trade",  # Sends to hub's trade level
                    "year_vtg": [year_vtg],
                    "year_act": [year_act],
                    "mode": "M1",
                    "time": "year",
                    "time_dest": "year",
                    "value": 1,
                    "unit": "-"
                })

# ======================
# 3. Hub Global Exporter
# ======================
for year_act in years:
    for year_vtg in years:
        if year_vtg <= year_act <= year_vtg + 40:
            # Hub -> Global transfer
            msg_scenario.add_par("output", {
                "node_loc": "Trade_hub",
                "node_dest": "global_trade",
                "technology": "lh2_global_exp",
                "commodity": "lh2",
                "level": "trade",  # Match global level
                "year_vtg": [year_vtg],
                "year_act": [year_act],
                "mode": "M1",
                "time": "year",
                "time_dest": "year",
                "value": 1,
                "unit": "-"
            })

            # Add corresponding input to hub transfer tech
            msg_scenario.add_par("input", {
                "node_loc": "Trade_hub",
                "node_origin": "Trade_hub",  # Takes from itself
                "technology": "lh2_global_exp",
                "commodity": "lh2",
                "level": "trade",  # Input from hub's trade level
                "year_vtg": [year_vtg],
                "year_act": [year_act],
                "mode": "M1",
                "time": "year",
                "time_origin": "year",
                "value": 1,
                "unit": "-"
            })

            

# ======================
# 4. Global Demand Setup
# ======================
global_demand = {
    2025: 5,    # GWa
    2030: 10,
    2040: 15,
    2045: 20,
    2050: 25,
}

for year, value in global_demand.items():
    msg_scenario.add_par(
        "demand",
        ["global_trade", "lh2", "trade", year, "year"],  # Key (5 elements)
        value,  # Separate value argument
        unit="GWa"
    )

# ======================
# 5. Technology Costs (Example)
# ======================
# Provincial export tech
for province in provinces:
    for year_vtg in years:
        msg_scenario.add_par("inv_cost", {
            "node_loc": [province],
            "technology": "lh2_exp",
            "year_vtg": [year_vtg],
            "value": 450,  # USD/kW
            "unit": "USD/kW"
        })

# Hub transfer tech
for year_vtg in years:
    msg_scenario.add_par("inv_cost", {
        "node_loc": "Trade_hub",
        "technology": "lh2_global_exp",
        "year_vtg": [year_vtg],
        "value": 0,  # Lower cost for hub-scale
        "unit": "USD/kW"
    })

tecs=['lh2_imp','lh2_exp']

for tec in tecs:   
    for year_act in years:
        for year_vtg in years:
            if year_vtg <= year_act <= year_vtg + 40:
                for province in provinces:    
                    cp = {
                        "node_loc": [province],
                        "technology": [tec],
                        "year_vtg": [year_vtg],
                        "year_act": [year_act],
                        "mode": ["M1"],
                        "time": ["year"],
                        "value": [1],
                        "unit": ["-"],
                    }
                    msg_scenario.add_par ("capacity_factor", cp)


for tec in tecs:
    for year_vtg in years:    
        for province in provinces:    
            inv_cost = {
                "node_loc": [province],
                "technology": [tec], 
                "year_vtg": [year_vtg],
                "value": [225],
                "unit": ["USD/kW"],
            }
            msg_scenario.add_par("inv_cost", inv_cost)

for tec in tecs:
    for year_vtg in years:    
        for province in provinces: 
            tec_lifetime = {
                "node_loc": [province],
                "technology": [tec], 
                "year_vtg": [year_vtg],
                "value": [40],
                "unit": ["year"],
            }
            msg_scenario.add_par("technical_lifetime", tec_lifetime)

for tec in tecs:
    for year_vtg in years:    
        for province in provinces:    
            inv_cost = {
                "node_loc": ["Trade_hub"],
                "technology": [tec], 
                "year_vtg": [year_vtg],
                "value": [225],
                "unit": ["USD/kW"],
            }
            msg_scenario.add_par("inv_cost", inv_cost)

for tec in tecs:
    for year_vtg in years:    
        for province in provinces: 
            tec_lifetime = {
                "node_loc": ["Trade_hub"],
                "technology": [tec], 
                "year_vtg": [year_vtg],
                "value": [40],
                "unit": ["year"],
            }
            msg_scenario.add_par("technical_lifetime", tec_lifetime)


for year_vtg in years:    
        tec_lifetime = {
                "node_loc": ["Trade_hub"],
                "technology": ["lh2_global_exp"], 
                "year_vtg": [year_vtg],
                "value": [40],
                "unit": ["year"],
            }
        msg_scenario.add_par("technical_lifetime", tec_lifetime)

for year_vtg in years:      
            inv_cost = {
                "node_loc": ["Trade_hub"],
                "technology": ["lh2_global_exp"], 
                "year_vtg": [year_vtg],
                "value": [225],
                "unit": ["USD/kW"],
            }
            msg_scenario.add_par("inv_cost", inv_cost)

for year_act in years:
    for year_vtg in years:
        if year_vtg <= year_act <= year_vtg + 40:
            msg_scenario.add_par("capacity_factor", {
                "node_loc": "Trade_hub",
                "technology": "lh2_global_exp",
                "year_vtg": [year_vtg],
                "year_act": [year_act],
                "mode": "M1",
                "time": "year",
                "value": 1,
                "unit": "-"
            })



In [ ]:
filtered_data = msg_scenario.par("technical_lifetime", filters={"technology": "lh2_imp"})
print(filtered_data)

In [ ]:
#This was used to fix the add_technology function

dfs = []

for year in years:
    emission_factor_df = message_ix.make_df(
        "relation_activity",
        node_rel=provinces,
        node_loc=provinces,
        technology=['h2_smr'] * len(provinces),
        mode=['M1'] * len(provinces),
        value=[0.5] * len(provinces),
        relation=['CO2_cc'] * len(provinces),
        unit=["-"] * len(provinces),
        year_rel=[year] * len(provinces),
        year_act=[year] * len(provinces),
    )
    dfs.append(emission_factor_df)

# Concatenate all DataFrames
combined_df = pd.concat(dfs, ignore_index=True)

# Add the combined DataFrame to the scenario
msg_scenario.add_par('relation_activity', combined_df)



In [ ]:
relation_activity_data = msg_scenario.par('relation_activity', filters={'technology': ['h2_pyrolysis'], 'relation': ['CO2_cc']})
print(relation_activity_data)


In [ ]:
'''provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

shares = ['share_h2_fc_i', 'share_h2_i', 'share_h2_fc_RC', 'share_h2_rc']

years = list(range(2025, 2061, 5))

# Get the entire share_commodity_lo parameter once
share_df = msg_scenario.par('share_commodity_lo')

# Rename columns to match expected dimensions
share_df = share_df.rename(columns={'node_share': 'node', 'year_act': 'year'})

# Print information about the DataFrame
print("\nDataFrame Info:")
share_df.info()

print("\nUnique values in 'shares' column:")
print(share_df['shares'].unique())

print("\nUnique values in 'node' column:")
print(share_df['node'].unique())

print("\nUnique values in 'year' column:")
print(share_df['year'].unique())

print("\nUnique values in 'time' column:")
print(share_df['time'].unique())

# Create a mask for the rows we want to modify
mask = (share_df['shares'].isin(shares)) & \
       (share_df['node'].isin(provinces)) & \
       (share_df['year'].isin(years)) & \
       (share_df['time'] == 'year')

# Print the number of True values in each part of the mask
print("\nNumber of True values in each part of the mask:")
print("shares mask:", share_df['shares'].isin(shares).sum())
print("node mask:", share_df['node'].isin(provinces).sum())
print("year mask:", share_df['year'].isin(years).sum())
print("time mask:", (share_df['time'] == 'year').sum())

# Multiply the values for the masked rows by 1.2
share_df.loc[mask, 'value'] *= 0.9

# Print summary to verify changes
print("\nSummary of modifications:")
print(share_df[mask]['value'].describe())

# Print the number of modified rows
print(f"\nNumber of rows modified: {mask.sum()}")

# Print unique values in 'shares' column to verify which shares were affected
print("\nShares affected:")
print(share_df[mask]['shares'].unique())

# Remove the existing parameter
msg_scenario.remove_par('share_commodity_lo')

msg_scenario.init_par('share_commodity_lo', ['shares', 'node', 'year', 'time'])

share_df = share_df.rename(columns={'node': 'node_share', 'year': 'year_act'})

# Add the modified parameter back to the scenario
msg_scenario.add_par('share_commodity_lo', share_df)'''

In [ ]:


'''years = list(range(2025, 2041, 5))
technology = "h2_atr_ccs"
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

# Loop through provinces and years to remove the 'input' parameter
for province in provinces:
    for year_act in years:
        for year_vtg in years:
            if year_vtg <= year_act <= year_vtg + 10:
                msg_scenario.remove_par('input', 
                    {
                        'node_loc': province,
                        'technology': technology,
                        'year_vtg': year_vtg,
                        'year_act': year_act,
                        'mode': "M1",
                        'node_origin': province,
                        'commodity': ["electr"],
                        'level': ["primary"],
                        'time': "year",
                        'time_origin': "year"
                    }
                )'''

In [ ]:
'''years = list(range(2020, 2041, 5))
technology = "h2_smr_ccs"
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

# Loop through provinces and years to remove the 'input' parameter
for province in provinces:
    for year_act in years:
        for year_vtg in years:
            if year_vtg <= year_act <= year_vtg + 25:
                msg_scenario.remove_par('emission_factor', 
                    {
                        'node_loc': province,
                        'technology': technology,
                        'year_vtg': year_vtg,
                        'year_act': year_act,
                        'mode': "M1",
                        'emission': ["CO2_Emission"],
                    }
                )'''

In [ ]:
'''import pandas as pd

# Define the provinces and the years
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

years = list(range(2030, 2061, 5))  # 2030, ..., 2060
years2 = list(range(2040, 2061, 5)) 
technologies = ['h2_atr_ccs', 'meth_h2']
technology2 = 'h2_gas_fc_trp'
value = 0.015
value2= 0.12162

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('initial_activity_up', 
                        [province, technology, year, "year"], 
                        value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years2:
        msg_scenario.add_par('initial_activity_up', 
                    [province, technology2, year, "year"], 
                    value2, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed       
    for technology in technologies:      
        msg_scenario.add_par('initial_activity_up', 
                    [province, technology, "2020", "year"], 
                    0.006, 'GWa')
    for technology in technologies:
        msg_scenario.add_par('initial_activity_up', 
                    [province, technology, "2025", "year"], 
                    0.0105, 'GWa')

    msg_scenario.add_par('initial_activity_up', 
                [province, technology2, "2025", "year"], 
                0.10815, 'GWa')

    msg_scenario.add_par('initial_activity_up', 
                [province, technology2, "2030", "year"], 
                0.11295, 'GWa')

    msg_scenario.add_par('initial_activity_up', 
                [province, technology2, "2035", "year"], 
                0.1173, 'GWa')'''



In [ ]:
value = 0.08
value2= 0.075
years = list(range(2020, 2061, 5)) 

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('growth_activity_up', 
                        [province, technology, year, "year"], 
                        value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed
        
        msg_scenario.add_par('growth_activity_up', 
                    [province, technology2, year, "year"], 
                    value2, 'GWa')

In [ ]:
value = 0.006
#h2_atr
technologies= ['h2_atr_ccs', 'meth_h2', 'h2_atr'] #meed to do for both w and without ccs

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('initial_new_capacity_up', 
                    [province, technology, year], 
                    value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed
            


In [ ]:
value = 0.05 #meed to do for both w and without ccs
# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('growth_new_capacity_up', 
                    [province, technology, year], 
                    value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed'''

In [18]:
technology = "h2_pyrolysis"
years = list(range(2020, 2061, 5)) 

for year in years:
    for province in provinces:
        msg_scenario.add_par('growth_new_capacity_up', 
                        [province, technology, year], 
                        0.05, 'GWa')
        
        msg_scenario.add_par('initial_new_capacity_up', 
                        [province, technology, year], 
                        0.006, 'GWa')
        
        msg_scenario.add_par('relation_activity', 
                        ["h2_exp_limit", province, year, province, technology, year, "M1"], 
                        1, 'GWa/a')
        
    



In [28]:
msg_scenario.add_set("cat_tec", ["transformation_fossil", "h2_atr_ccs"])

msg_scenario.add_set("cat_tec", ["powerplant", "h2_atr_ccs"])

In [ ]:
'''msg_scenario.add_set("cat_tec", ["h2_trp", "h2_gas_fc_trp"])

msg_scenario.add_set("cat_tec", ["transport", "h2_gas_fc_trp"])

# Optionally, you can check if the addition was successful
cat_tec = msg_scenario.set("cat_tec")
print(cat_tec)'''

In [27]:
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']
years = list(range(2020, 2061, 5)) 
value = -0.62319
value2= 1

for province in provinces:
    for year in years:
      msg_scenario.add_par('relation_activity', 
                    ["h2_scrub_limit", province, year, province, "h2_atr_ccs", year, "M1"], 
                    value, 'GWa/a') 
      
      msg_scenario.add_par('relation_activity', 
                    ["h2_exp_limit", province, year, province, "h2_atr_ccs", year, "M1"], 
                    value2, 'GWa/a')
      msg_scenario.add_par('relation_activity', 
                    ["h2_exp_limit", province, year, province, "h2_atr", year, "M1"], 
                    value2, 'GWa/a') 
      

In [47]:
msg_scenario.commit("added parameters/sets")

In [12]:
msg_scenario.solve(
					solve_options={"barcrossalg": "2", "lpmethod": "1"})

In [ ]:

"""
Apply the Output-Based Pricing System (OBPS) policy for the Baseline Canada Scenario
to the given MESSAGEix scenario.

This function modifies the scenario's fixed costs based on emissions intensity and carbon pricing.

Args:
    scenario (ixmp.Scenario): The MESSAGEix scenario to apply the OBPS policy to.
    data_file_path (str): Path to the Excel file containing necessary data.
"""

'''
# Read the required data
data_relation = pd.read_excel("model/scenarios/baseline/MESSAGE_Updated_LR.xlsx", sheet_name='emission_factor')

provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
            'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
            'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

# Define conditions for filtering the data
conditions_2 = (
    data_relation['technology'].isin([
        'oil_extr_1', 'oil_extr_2', 'oil_extr_3', 'oil_extr_4', 'oil_extr_5', 'oil_extr_6',
        'cement_CO2', 'coal_extr', 'gas_extr_1', 'gas_extr_2', 'gas_extr_3', 'gas_extr_4', 
        'gas_extr_5', 'gas_extr_6', 'h2_smr_ccs', 'h2_smr', 'h2_atr', 'h2_atr_ccs'
    ]) &
    (data_relation['emission'] == 'CO2_cc') &
    (data_relation['node_loc'].isin(provinces))
)

# Filter the data based on the conditions
result = data_relation[conditions_2][['technology', 'value', 'node_loc', 'year_act']]

# Initialize columns for calculations
result['Emission Intensity Reduction_baseline'] = 0.0

# Calculate Emission Intensity Reduction_baseline
for index, row in result.iterrows():
    if 2020 < row['year_act'] < 2055:
        previous_year_value = result.loc[
            (result['technology'] == row['technology']) & 
            (result['year_act'] == row['year_act'] - 5), 
            'Emission Intensity Reduction_baseline'
        ].values
        if len(previous_year_value) > 0:
            result.loc[index, 'Emission Intensity Reduction_baseline'] += previous_year_value[0] + 3.33/100

# Calculate Emission intensity standard_Baseline Canada Scenario
result['Emission intensity standard_Baseline Canada Scenario'] = 0.0
for index, row in result.iterrows():
    if row['year_act'] > 2020:
        value_cal = result.loc[
            (result['technology'] == row['technology']) & 
            (result['year_act'] == 2015), 
            'value'
        ]
        if not value_cal.empty:
            result.at[index, 'Emission intensity standard_Baseline Canada Scenario'] = row['value'] - row['Emission Intensity Reduction_baseline'] * value_cal.values[0]
        else:
            value_cal = result.loc[
                (result['technology'] == row['technology']) & 
                (result['year_act'] == 2020), 
                'value'
            ]
            result.at[index, 'Emission intensity standard_Baseline Canada Scenario'] = row['value'] - row['Emission Intensity Reduction_baseline'] * value_cal.values[0]

# Calculate Payable emission intensity_Baseline scenario
result['Payable emission intensity_Baseline scenario'] = 0.0
for index, row in result.iterrows():
    if row['year_act'] > 2020:
        result.loc[index, 'Payable emission intensity_Baseline scenario'] = row['value'] - row['Emission intensity standard_Baseline Canada Scenario']

# Calculate carbon cost_Baseline Canada scenario
rate = 0.73
result['carbon cost_Baseline Canada scenario'] = 0.0
for index, row in result.iterrows():
    if row['year_act'] == 2025:
        result.loc[index, 'carbon cost_Baseline Canada scenario'] = row['Payable emission intensity_Baseline scenario'] * rate * 125
    elif row['year_act'] > 2025:
        result.loc[index, 'carbon cost_Baseline Canada scenario'] = row['Payable emission intensity_Baseline scenario'] * rate * 170

# Adjust for years 2055 and 2060
for index, row in result.iterrows():
    if row['year_act'] > 2050:
        value_to_assign = result.loc[
            (result['technology'] == row['technology']) & 
            (result['year_act'] == 2050), 
            'carbon cost_Baseline Canada scenario'
        ].values
        if len(value_to_assign) > 0:
            result.at[index, 'carbon cost_Baseline Canada scenario'] = value_to_assign[0]

# Merge with fix_cost data and update the scenario's fix_cost
data_relation = pd.read_excel("model/scenarios/baseline/MESSAGE_Updated_LR.xlsx", sheet_name='emission_factor')
# Read the fix_cost data
data_fixcost = pd.read_excel("model/scenarios/baseline/MESSAGE_Updated_LR.xlsx", sheet_name='fix_cost')

merged_data = pd.merge(
    data_fixcost, 
    result[['technology', 'year_act', 'node_loc', 'carbon cost_Baseline Canada scenario']],
    left_on=['technology', 'year_act', 'node_loc'],
    right_on=['technology', 'year_act', 'node_loc'],
    how='left'
)
merged_data['value'] = merged_data['value'] + merged_data['carbon cost_Baseline Canada scenario'].fillna(0)
merged_data.drop(columns=['carbon cost_Baseline Canada scenario'], inplace=True)

# Assuming 'merged_data' contains the data you've shown

for index, row in merged_data.iterrows():
    msg_scenario.add_par('fix_cost', pd.DataFrame({
        'node_loc': [row['node_loc']],
        'technology': [row['technology']],
        'year_vtg': [row['year_vtg']],
        'year_act': [row['year_act']],
        'mode': ['M1'],  # Assuming 'standard' mode for all entries
        'unit': [row['unit']],
        'value': [row['value']]
    }, index=[0]))'''

In [ ]:
import pandas as pd

In [ ]:

df_up = msg_scenario.par('bound_activity_up')
df_up = df_up[df_up['technology'].str.contains('gas_extr|oil_extr')]
# Identifying unique node_loc and technology combinations
unique_combinations = df_up[["node_loc", "technology"]].drop_duplicates()

# Creating new rows for 2055 and 2060 by copying 2050 values
new_rows = []
for _, row in df_up[df_up["year_act"] == 2050].iterrows():
    for new_year in [2055, 2060]:
        new_row = row.copy()
        new_row["year_act"] = new_year
        new_rows.append(new_row)


df_extended_lo = pd.concat([df_up, pd.DataFrame(new_rows)], ignore_index=True)

msg_scenario.add_par('bound_activity_up', df_extended_lo)

In [ ]:
df_up['technology'].unique()

In [ ]:
df_extended_lo[df_extended_lo['year_act'] >2050]

In [48]:
msg_scenario.to_excel(r"C:\Users\brend\Documents\GitHub\message-ix - CA\model\notebooks\netzero_cost_test.xlsx")


In [ ]:
output_path=r"C:/Users/brend/Documents/Github/message-ix - CA/model/scenarios/baseline/MESSAGE_baseline_obps.csv"
min_year= 2015
max_year= 2060
from_excel=False
unit_correct=True

rep = message_ix.Reporter.from_scenario(
    msg_scenario,
    units={"apply": {"var_cost": "", "inv_cost": "", "fix_cost": ""}},
)
report = rep.get("message::default")

if from_excel:
    report_df = pd.read_csv("MESSAGE_NO_example_default_report.csv")
else:
    report_df = report.timeseries()
    report_df.reset_index(inplace=True)
    report_df.columns = report_df.columns.astype(str)
    report_df.columns = report_df.columns.str.title()

years = msg_scenario.set("year").tolist()
years = [str(x) for x in years if x >= min_year and x <= max_year]

report_df.fillna(0, inplace=True)

report_df.columns = report_df.columns.astype(str)
report_df = report_df.drop(
    report_df.columns.to_series()["2020" : str(min_year)], axis=1
)
report_df.to_csv('test.csv')
in_flows = report_df[report_df["Variable"].str.startswith("in|")]
out_flows = report_df[report_df["Variable"].str.startswith("out|")]
all_flows = pd.concat([out_flows, in_flows])

report_df["Model"] = "MESSAGEix-Canada"
report_df["Scenario"] = "Reference"
report_df["Variable"] = report_df["Variable"].str.replace(
    "emis", "Emissions", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|final|", "Final Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|final|", "Final Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|primary|", "Primary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|primary|", "Primary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|secondary|", "Secondary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|secondary|", "Secondary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|renewable|", "Primary Energy|renewable|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|renewable|", "Primary Energy|renewable|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "total om cost|", "Total Costs|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|useful|", "Useful Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|useful|", "Useful Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "CAP_NEW|new capacity|", "Capacity Addition|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "CAP|capacity|", "Capacity|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "inv cost|", "Capital Cost|", regex=False
)
report_df["Region"] = report_df["Region"].apply(lambda x: x.split("|")[0])

report_df.columns = report_df.columns.str.strip()  # Remove any leading/trailing whitespace
report_df.rename(columns={'Model': 'model', 'Scenario': 'scenario', 'Variable': 'variable'}, inplace=True)

report_df = report_df.drop_duplicates(subset=["model", "scenario", "Region", "variable"], keep='first')


pyam_df = pyam.IamDataFrame(report_df)

variable = pyam_df.variable
pyam_df.aggregate_region(variable, append=True)
pyam_df.rename({"region": {"World": "Canada"}}, inplace=True)

pyam_df.to_csv(output_path)

print(f"Report saved to {output_path}")



In [ ]:
'''import os

output_dir = r'C:\Users\brend\Documents\Github\message-ix - CA\scenarios\baseline'

# Check if the directory exists, if not, create it
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'report.csv')

# Now save the file
pyam_df.to_csv(output_path)
print(f"Report saved to {output_path}")'''

In [ ]:
print(report_df["Region"].value_counts())

In [ ]:
report_df[report_df["variable"] =='Secondary Energy|electr|h2_smr_ccs|M1']


In [ ]:
print(report_df["variable"].value_counts())

In [ ]:
report_df.columns

In [ ]:
test = report_df['variable'].value_counts()
# print("Number of unique variables:", report_df['Variable'].nunique())


In [ ]:
#from message_ix_models.tools.costs.config import Config
#from message_ix_models.tools.costs.projections import create_cost_projections

In [ ]:
import pandas as pd

def update_inv_cost_with_fallback(scenario):
    reduction_2100_percent = {
        'coal_adv_ccs': 0.50,
        'igcc_ccs': 0.50,
        'gas_cc_ccs': 0.38,
        'bio_istig_ccs': 0.40,
        'syn_liq_ccs': 0.25,
        'meth_coal_ccs': 0.15,
        'h2_coal': 0.40,
        'h2_smr': 0.50,
        'h2_bio': 0.50,
        'h2_coal_ccs': 0.50,
        'h2_smr_ccs': 0.50,
        'h2_bio_ccs': 0.50,
        'eth_bio_ccs': 0.40,
        'meth_ng_ccs': 0.15,
        'h2_elec': 0.20,
        'liq_bio_ccs': 0.40,
        'h2_atr': 0.50,
        'h2_atr_ccs': 0.50,
        'h2_stor_geo': 0.15,
        'liq_h2': 0.20,
        'meth_h2': 0.20,
        'h2_fc_trp': 0.56,
        'h2_gas_fc_trp': 0.56,
        'gas_h2': 0.20
    }

    # Configuration - easily adjustable if needed
    BASE_YEARS = [2020, 2025]  # Priority order
    FINAL_YEAR = 2100
    MODEL_END_YEAR = 2060

    # Get available model years
    model_years = sorted([
        int(y) for y in scenario.set('year_vtg') 
        if int(y) >= BASE_YEARS[-1] and int(y) <= MODEL_END_YEAR
    ])

    # Load investment costs
    inv_cost = scenario.par('inv_cost')
    updated_rows = []

    for tech, reduction in reduction_2100_percent.items():
        base_cost = None
        base_year = None
        
        # Find first available base year
        for candidate_year in BASE_YEARS:
            tech_data = inv_cost[
                (inv_cost['technology'] == tech) & 
                (inv_cost['year_vtg'] == candidate_year)
            ]
            if not tech_data.empty:
                base_year = candidate_year
                base_cost = tech_data['value'].values[0]
                break

        if base_cost is None:
            print(f"Skipping {tech}: No inv_cost found in {BASE_YEARS}")
            continue

        # Calculate annual reduction factor
        final_cost = base_cost * (1 - reduction)
        annual_factor = (final_cost / base_cost) ** (1/(FINAL_YEAR - base_year))

        # Apply to all model years >= base_year
        for year in model_years:
            if year < base_year:
                continue
                
            years_since_base = year - base_year
            new_cost = base_cost * (annual_factor ** years_since_base)
            
            # Preserve all original columns
            new_row = tech_data.iloc[0].copy()
            new_row.update({
                'year_vtg': year,
                'value': new_cost
            })
            updated_rows.append(new_row)

    # Update scenario
    if updated_rows:
        new_inv_cost = pd.DataFrame(updated_rows)
        scenario.add_par('inv_cost', new_inv_cost)
        print(f"Successfully updated {len(new_inv_cost)} entries")
    else:
        print("No updates applied")

    return scenario

 

In [ ]:

import pandas as pd

def apply_learning_rates(msg_scenario, base_year, learning_rates, last_year=2060, step=5):
    """
    Update inv_cost parameter for specified technologies using learning rates.

    Args:
        scenario: message_ix.Scenario object (already loaded)
        base_year: int, e.g., 2025
        learning_rates: dict of {technology: fractional_rate}, e.g., {'coal_adv_ccs': 0.5}
        last_year: int, last year to apply learning
        step: int, time step in years
    """
    # Get investment cost parameter as DataFrame
    inv_cost = msg_scenario.par('inv_cost')
    periods = sorted([y for y in msg_scenario.set('year') if int(y) >= base_year and int(y) <= last_year])
    df_list = []

    for tech, rate in learning_rates.items():
        tech_df = inv_cost[inv_cost['technology'] == tech].copy()
        # Find all nodes for this technology
        for node in tech_df['node_loc'].unique():
            node_df = tech_df[tech_df['node_loc'] == node]
            # Get base cost for base_year
            base_row = node_df[node_df['year_vtg'] == base_year]
            if base_row.empty:
                continue
            base_cost = base_row.iloc[0]['value']
            # Apply learning for each period
            for year in periods:
                new_row = base_row.iloc[0].copy()
                new_row['year_vtg'] = year
                years_since_base = int(year) - base_year
                n_steps = years_since_base // step  # step = 5 for 5-year increments
                reduction_factor = (1 - rate) ** n_steps
                new_row['value'] = base_cost * reduction_factor
                df_list.append(pd.DataFrame([new_row]))


    # Combine all new rows
    new_inv_cost = pd.concat(df_list, ignore_index=True)
    # Remove old entries for these technologies and years
    mask = inv_cost['technology'].isin(learning_rates.keys()) & inv_cost['year_vtg'].isin(periods)
    inv_cost = inv_cost[~mask]
    # Add new calculated values
    inv_cost = pd.concat([inv_cost, new_inv_cost], ignore_index=True)
    # Remove and re-add parameter
    #msg_scenario.remove_par('inv_cost')
    msg_scenario.add_par('inv_cost', inv_cost)
    print("Investment costs updated with learning rates.")

# Define your learning rates (fractional, e.g., 0.5 for 50%)
learning_rates = {
    'coal_adv_ccs': 0.50,
    'igcc_ccs': 0.50,
    'gas_cc_ccs': 0.38,
    'bio_istig_ccs': 0.40,
    'syn_liq_ccs': 0.25,
    'meth_coal_ccs': 0.15,
    'h2_coal': 0.40,
    'h2_smr': 0.50,
    'h2_bio': 0.50,
    'h2_coal_ccs': 0.50,
    'h2_smr_ccs': 0.50,
    'h2_bio_ccs': 0.50,
    'eth_bio_ccs': 0.40,
    'meth_ng_ccs': 0.15,
    'h2_elec': 0.20,
    'liq_bio_ccs': 0.40,
    'h2_atr': 0.50,
    'h2_atr_ccs': 0.50,
    #'h2_stor_geo': 0.15,
    #'liq_h2': 0.20,
    #'meth_h2': 0.20,
    #'h2_fc_trp': 0.56,
    #'h2_gas_fc_trp': 0.56,
    #'gas_h2': 0.20,
}

# Example usage:
# scenario = Scenario(platform, model, scenario_name)
# apply_learning_rates(scenario, base_year=2025, learning_rates=learning_rates)
# scenario.commit("Updated inv_cost with technology learning rates")


In [36]:
apply_learning_rates(msg_scenario, base_year=2025, learning_rates=learning_rates)

Investment costs updated with learning rates.


In [ ]:
msg_scenario.solve(
					solve_options={"barcrossalg": "2", "lpmethod": "1"})

In [ ]:
msg_plt = Plotter_MSG(msg_scenario)
nodes = msg_scenario.set('node')[2:]


In [ ]:
msg_plt.plotter(case="test",nodeloc=nodes,path="output",yr_min=2020, yr_max=2060)